In [ ]:
from pathlib import Path
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from inference import ConditionSpec, PredictConfig, predict_with_config
from models import DDPM

SEED = 0


In [ ]:
class SmokeVAE(nn.Module):
    def encode(self, x):
        mu = F.avg_pool2d(x, kernel_size=4, stride=4).repeat(1, 4, 1, 1)
        return mu, torch.zeros_like(mu)

    def decode(self, z):
        return F.interpolate(z[:, :1], scale_factor=4, mode="nearest").clamp(0, 1)


class SmokeConditionUNet(nn.Module):
    def forward(self, z_t, t, condition_z, axis, slice_index):
        return torch.zeros_like(z_t)


torch.manual_seed(SEED)
vae = SmokeVAE()
unet = SmokeConditionUNet()
ddpm = DDPM(timesteps=3)


In [ ]:
condition = torch.ones(1, 1, 64, 64)
config = PredictConfig(
    conditions=[
        ConditionSpec(condition=condition, axis=0, slice_index=12),
        ConditionSpec(condition=condition, axis=1, slice_index=8),
    ],
    volume_shape=(4, 16, 16, 16),
)
result = predict_with_config(
    vae=vae,
    unet=unet,
    ddpm=ddpm,
    config=config,
)

z_error, y_error = result["condition_errors"]
volume = result["volume"]
lock_z = (volume[12] - condition.squeeze(0)).abs().max()
lock_y = (volume[:, 0, 8, :] - 1).abs().max()
volume.shape, z_error, y_error, float(lock_z), float(lock_y)
